In [1]:
!pip install -q llama-index
!pip install torch transformers scipy scikit-learn

In [5]:
import sys
from pathlib import Path
import pickle
import time
import logging
import numpy as np
import torch
from scipy.sparse import csr_matrix, save_npz
from transformers import AutoModelForMaskedLM, AutoTokenizer
from llama_index.core import Document
from typing import List, Union

# Add parent directory to Python path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents

class SpladeSparseVectorizer:
    """
    SPLADE (SParse Lexical AnD Expansion) vectorizer for Spanish documents.
    Uses a pre-trained Spanish BERT model to create sparse document representations.
    """
    
    def __init__(self, model_name="dccuchile/bert-base-spanish-wwm-cased", 
                 max_length=256, batch_size=2, device=None):
        """
        Initialize SPLADE vectorizer with conservative defaults for memory.
        
        Args:
            model_name: HuggingFace model name for Spanish BERT
            max_length: Maximum sequence length for tokenization (reduced default)
            batch_size: Batch size for processing documents (reduced default)
            device: Device to run the model on (auto-detected if None)
        """
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        
        # Set device
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        
        print(f"Using device: {self.device}")
        
        # Load tokenizer and model with memory optimizations
        print(f"Loading tokenizer and model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Load model with consistent precision
        if self.device.type == 'cuda':
            self.model = AutoModelForMaskedLM.from_pretrained(
                model_name, 
                torch_dtype=torch.float32  # Use float32 for consistency
            )
        else:
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)
            
        self.model.to(self.device)
        self.model.eval()
        
        # Vocabulary size
        self.vocab_size = len(self.tokenizer.vocab)
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Model loaded with batch_size={batch_size}, max_length={max_length}")
        
        # Document statistics (will be computed during fit)
        self.fitted = False
        
    def _extract_splade_features(self, texts: List[str]) -> csr_matrix:
        """
        Extract SPLADE sparse features from texts with memory optimization.
        
        Args:
            texts: List of text strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        # Use lists to store sparse matrix components efficiently
        rows, cols, data = [], [], []
        current_row = 0
        
        print(f"Processing {len(texts)} texts in batches of {self.batch_size}...")
        batch_start_time = time.time()
        
        # Process in batches with progress tracking
        for batch_idx in range(0, len(texts), self.batch_size):
            batch_texts = texts[batch_idx:batch_idx + self.batch_size]
            
            # Enhanced progress tracking for large datasets
            current_batch = batch_idx // self.batch_size + 1
            total_batches = (len(texts) - 1) // self.batch_size + 1
            
            if current_batch % max(1, total_batches // 20) == 0 or current_batch == 1:  # Show progress every 5%
                progress = (current_batch / total_batches) * 100
                print(f"Progress: {progress:.1f}% - Batch {current_batch}/{total_batches}")
                
                # Estimate remaining time
                if current_batch > 1:
                    elapsed = time.time() - batch_start_time
                    avg_time_per_batch = elapsed / (current_batch - 1)
                    remaining_batches = total_batches - current_batch
                    est_remaining = avg_time_per_batch * remaining_batches
                    minutes, seconds = divmod(est_remaining, 60)
                    print(f"Estimated time remaining: {int(minutes)}m {int(seconds)}s")
            
            try:
                # Tokenize batch
                inputs = self.tokenizer(
                    batch_texts,
                    max_length=self.max_length,
                    padding=True,
                    truncation=True,
                    return_tensors="pt"
                ).to(self.device)
                
                with torch.no_grad():
                    # Get model outputs
                    outputs = self.model(**inputs)
                    logits = outputs.logits  # [batch_size, seq_len, vocab_size]
                    
                    # Apply log(1 + ReLU) activation to get sparse weights
                    sparse_weights = torch.log1p(torch.relu(logits))
                    
                    # Max pooling over sequence dimension to get document representation
                    # Shape: [batch_size, vocab_size]
                    doc_sparse = torch.max(sparse_weights, dim=1)[0]
                    
                    # Process each document in the batch
                    for batch_doc_idx, doc_weights in enumerate(doc_sparse):
                        # Apply threshold to reduce sparsity and memory usage
                        threshold = 0.1  # Only keep values above this threshold
                        significant_mask = doc_weights > threshold
                        
                        if significant_mask.any():
                            # Get indices and values for significant weights
                            indices = torch.nonzero(significant_mask).squeeze(-1)
                            values = doc_weights[indices]
                            
                            # Convert to CPU and add to sparse matrix components
                            indices_cpu = indices.cpu().numpy().astype(np.int32)
                            values_cpu = values.cpu().numpy().astype(np.float32)
                            
                            # Add to sparse matrix components
                            rows.extend([current_row] * len(indices_cpu))
                            cols.extend(indices_cpu.tolist())
                            data.extend(values_cpu.tolist())
                        
                        current_row += 1
                
                # Clear GPU memory after each batch
                del inputs, outputs, logits, sparse_weights, doc_sparse
                torch.cuda.empty_cache() if torch.cuda.is_available() else None
                
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"GPU memory error at batch {batch_idx // self.batch_size + 1}. Trying to continue...")
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
                    # Try processing with smaller batch size
                    return self._extract_splade_features_fallback(texts)
                else:
                    raise e
        
        # Create sparse matrix from components with explicit dtype
        if rows:
            # Ensure all components are the right type
            rows = np.array(rows, dtype=np.int32)
            cols = np.array(cols, dtype=np.int32)
            data = np.array(data, dtype=np.float32)
            
            sparse_matrix = csr_matrix((data, (rows, cols)), 
                                     shape=(len(texts), self.vocab_size),
                                     dtype=np.float32)
        else:
            # Handle edge case of no features
            sparse_matrix = csr_matrix((len(texts), self.vocab_size), dtype=np.float32)
        
        print(f"Created sparse matrix with shape: {sparse_matrix.shape}")
        print(f"Non-zero elements: {sparse_matrix.nnz}")
        
        return sparse_matrix
    
    def _extract_splade_features_fallback(self, texts: List[str]) -> csr_matrix:
        """
        Fallback method with smaller batch size for memory issues.
        """
        print("Using fallback method with batch size 1...")
        original_batch_size = self.batch_size
        self.batch_size = 1
        
        try:
            result = self._extract_splade_features(texts)
            return result
        finally:
            self.batch_size = original_batch_size
    
    def fit(self, documents: List[Union[Document, str]]):
        """
        Fit the SPLADE vectorizer to documents.
        
        Args:
            documents: List of Document objects or strings
        """
        print("Extracting text from documents...")
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        
        print(f"Fitting SPLADE vectorizer on {len(texts)} documents...")
        
        # For SPLADE, we don't need to compute global statistics like BM25
        # The model learns the sparse representations end-to-end
        self.fitted = True
        print("SPLADE vectorizer fitted successfully.")
        
        return self
    
    def transform(self, documents: List[Union[Document, str]]) -> csr_matrix:
        """
        Transform documents to SPLADE sparse representations.
        
        Args:
            documents: List of Document objects or strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        if not self.fitted:
            raise ValueError("Vectorizer must be fitted before transforming documents.")
        
        if isinstance(documents, str):
            documents = [documents]
        
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        
        print(f"Transforming {len(texts)} documents to SPLADE representations...")
        sparse_matrix = self._extract_splade_features(texts)
        
        print(f"Generated sparse matrix with shape: {sparse_matrix.shape}")
        print(f"Sparsity: {1 - sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]):.4f}")
        
        return sparse_matrix

def save_index(index_path: Path, vectorizer: SpladeSparseVectorizer, embeddings: csr_matrix, documents: List):
    """
    Save SPLADE index components to disk.
    
    Args:
        index_path: Path to save the index
        vectorizer: Fitted SPLADE vectorizer
        embeddings: Sparse embeddings matrix
        documents: Original documents
    """
    index_path.mkdir(parents=True, exist_ok=True)
    
    # Save vectorizer (without the model to save space)
    vectorizer_state = {
        'model_name': vectorizer.model_name,
        'max_length': vectorizer.max_length,
        'batch_size': vectorizer.batch_size,
        'vocab_size': vectorizer.vocab_size,
        'fitted': vectorizer.fitted
    }
    
    with open(index_path / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer_state, f)
    
    # Save embeddings
    save_npz(index_path / "embeddings.npz", embeddings)
    
    # Save documents
    with open(index_path / "documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    
    logging.info(f"SPLADE index saved to {index_path}")

def load_splade_vectorizer(index_path: Path) -> SpladeSparseVectorizer:
    """
    Load a saved SPLADE vectorizer.
    
    Args:
        index_path: Path where the index was saved
        
    Returns:
        Loaded SPLADE vectorizer
    """
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer_state = pickle.load(f)
    
    # Recreate vectorizer
    vectorizer = SpladeSparseVectorizer(
        model_name=vectorizer_state['model_name'],
        max_length=vectorizer_state['max_length'],
        batch_size=vectorizer_state['batch_size']
    )
    vectorizer.fitted = vectorizer_state['fitted']
    
    return vectorizer

def main(file: str, model_name: str = "dccuchile/bert-base-spanish-wwm-cased", 
         max_length: int = 512, batch_size: int = 2, chunk_size: int = 5000):
    """
    Main function to create SPLADE indices for Spanish documents.
    Memory-optimized version with chunked processing for large datasets.
    
    Args:
        file: Base filename for the document files
        model_name: HuggingFace model name for Spanish BERT
        max_length: Maximum sequence length (reduced default for memory)
        batch_size: Batch size for processing (reduced default for memory)
        chunk_size: Process documents in chunks to manage memory
    """
    start = time.time()
    print("Starting SPLADE indexing process for Spanish documents...")
    print(f"Using settings: max_length={max_length}, batch_size={batch_size}, chunk_size={chunk_size}")
    
    # Setup logging
    logging.basicConfig(level=logging.INFO)
    
    print("Setting up file paths...")
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')
    
    # Create index paths with model info
    model_suffix = model_name.split('/')[-1]
    index_path_texto = Path(f'/work/sandbox/splade/index/{model_suffix}_texto_full')
    index_path_resumen = Path(f'/work/sandbox/splade/index/{model_suffix}_resumen_full')
    
    print("Loading documents for 'texto'...")
    documents_texto = load_documents(documents_path_texto)
    print("Loading documents for 'resumen'...")
    documents_resumen = load_documents(documents_path_resumen)
    
    if documents_texto is None or documents_resumen is None:
        logging.error("Failed to load one or both document sets. Exiting.")
        return
    
    print(f"Processing all documents - texto: {len(documents_texto)}, resumen: {len(documents_resumen)}")
    
    datasets = [
        ("texto", documents_texto, index_path_texto),
        ("resumen", documents_resumen, index_path_resumen)
    ]
    total_steps = len(datasets)
    
    for i, (name, docs, index_path) in enumerate(datasets, 1):
        print(f"\nProcessing {name} documents ({i}/{total_steps})...")
        print(f"Document count: {len(docs)}")
        step_start = time.time()
        
        try:
            # Process in chunks to manage memory
            process_dataset_in_chunks(docs, index_path, model_name, max_length, batch_size, chunk_size, name)
            
        except Exception as e:
            print(f"Error processing {name} documents: {str(e)}")
            print("Skipping this dataset and continuing...")
            continue
        
        step_end = time.time()
        step_elapsed = step_end - step_start
        remaining_steps = total_steps - i
        estimated_remaining = step_elapsed * remaining_steps if remaining_steps > 0 else 0
        minutes, seconds = divmod(estimated_remaining, 60)
        print(f"Completed processing {name} documents in {step_elapsed:.2f} seconds. "
              f"Estimated time remaining: {int(minutes)} minutes, {int(seconds)} seconds.")
    
    end = time.time()
    total_time = end - start
    print(f"\nTotal execution time: {total_time:.2f} seconds")
    print("SPLADE indexing completed successfully!")

def process_dataset_in_chunks(documents, index_path, model_name, max_length, batch_size, chunk_size, dataset_name):
    """
    Process a large dataset in chunks to manage memory usage.
    """
    from scipy.sparse import vstack as sparse_vstack
    
    total_docs = len(documents)
    num_chunks = (total_docs + chunk_size - 1) // chunk_size
    
    print(f"Processing {total_docs} documents in {num_chunks} chunks of {chunk_size}")
    
    all_embeddings = []
    chunk_start_time = time.time()
    
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_docs)
        chunk_docs = documents[start_idx:end_idx]
        
        print(f"\nChunk {chunk_idx + 1}/{num_chunks}: Processing documents {start_idx}-{end_idx}")
        
        # Initialize vectorizer for this chunk
        vectorizer = SpladeSparseVectorizer(
            model_name=model_name,
            max_length=max_length,
            batch_size=batch_size
        )
        
        # Fit and transform this chunk
        print("Fitting and transforming chunk...")
        vectorizer.fit(chunk_docs)
        chunk_embeddings = vectorizer.transform(chunk_docs)
        
        # Store embeddings
        all_embeddings.append(chunk_embeddings)
        
        # Clean up memory
        del vectorizer, chunk_embeddings
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        # Progress and time estimation
        chunk_elapsed = time.time() - chunk_start_time
        avg_time_per_chunk = chunk_elapsed / (chunk_idx + 1)
        remaining_chunks = num_chunks - (chunk_idx + 1)
        est_remaining = avg_time_per_chunk * remaining_chunks
        minutes, seconds = divmod(est_remaining, 60)
        
        print(f"Chunk completed. Estimated time remaining: {int(minutes)}m {int(seconds)}s")
    
    # Combine all embeddings
    print("Combining all chunk embeddings...")
    final_embeddings = sparse_vstack(all_embeddings)
    
    # Clear chunk embeddings from memory
    del all_embeddings
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Create a final vectorizer for saving (using first chunk settings)
    print("Creating final vectorizer for saving...")
    final_vectorizer = SpladeSparseVectorizer(
        model_name=model_name,
        max_length=max_length,
        batch_size=batch_size
    )
    final_vectorizer.fit(documents[:10])  # Fit on small sample just to set state
    
    # Save the complete index
    print(f"Saving complete SPLADE index for {dataset_name}...")
    save_index(index_path, final_vectorizer, final_embeddings, documents)
    
    # Final cleanup
    del final_vectorizer, final_embeddings
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    print(f"Successfully processed {total_docs} documents in chunks!")

In [6]:
main('OEB', "dccuchile/bert-base-spanish-wwm-cased", 512, 2)

Starting SPLADE indexing process for Spanish documents...
Using settings: max_length=512, batch_size=2, chunk_size=5000
Setting up file paths...
Loading documents for 'texto'...


INFO:root:Successfully loaded 47514 documents from /work/data/processed/OEB_texto.pkl


Loading documents for 'resumen'...


INFO:root:Successfully loaded 47514 documents from /work/data/processed/OEB_resumen.pkl


Processing all documents - texto: 47514, resumen: 47514

Processing texto documents (1/2)...
Document count: 47514
Processing 47514 documents in 10 chunks of 5000

Chunk 1/10: Processing documents 0-5000
Using device: cuda
Loading tokenizer and model: dccuchile/bert-base-spanish-wwm-cased


Vocabulary size: 31002
Model loaded with batch_size=2, max_length=512
Fitting and transforming chunk...
Extracting text from documents...
Fitting SPLADE vectorizer on 5000 documents...
SPLADE vectorizer fitted successfully.
Transforming 5000 documents to SPLADE representations...
Processing 5000 texts in batches of 2...
Progress: 0.0% - Batch 1/2500
Progress: 5.0% - Batch 125/2500
Estimated time remaining: 0m 37s
Progress: 10.0% - Batch 250/2500
Estimated time remaining: 0m 33s
Progress: 15.0% - Batch 375/2500
Estimated time remaining: 0m 32s
Progress: 20.0% - Batch 500/2500
Estimated time remaining: 0m 29s
Progress: 25.0% - Batch 625/2500
Estimated time remaining: 0m 27s
Progress: 30.0% - Batch 750/2500
Estimated time remaining: 0m 26s
Progress: 35.0% - Batch 875/2500
Estimated time remaining: 0m 24s
Progress: 40.0% - Batch 1000/2500
Estimated time remaining: 0m 22s
Progress: 45.0% - Batch 1125/2500
Estimated time remaining: 0m 20s
Progress: 50.0% - Batch 1250/2500
Estimated time rema

INFO:root:SPLADE index saved to /work/sandbox/splade/index/bert-base-spanish-wwm-cased_texto_full


Successfully processed 47514 documents in chunks!
Completed processing texto documents in 983.53 seconds. Estimated time remaining: 16 minutes, 23 seconds.

Processing resumen documents (2/2)...
Document count: 47514
Processing 47514 documents in 10 chunks of 5000

Chunk 1/10: Processing documents 0-5000
Using device: cuda
Loading tokenizer and model: dccuchile/bert-base-spanish-wwm-cased
Vocabulary size: 31002
Model loaded with batch_size=2, max_length=512
Fitting and transforming chunk...
Extracting text from documents...
Fitting SPLADE vectorizer on 5000 documents...
SPLADE vectorizer fitted successfully.
Transforming 5000 documents to SPLADE representations...
Processing 5000 texts in batches of 2...
Progress: 0.0% - Batch 1/2500
Progress: 5.0% - Batch 125/2500
Estimated time remaining: 0m 29s
Progress: 10.0% - Batch 250/2500
Estimated time remaining: 0m 27s
Progress: 15.0% - Batch 375/2500
Estimated time remaining: 0m 25s
Progress: 20.0% - Batch 500/2500
Estimated time remaining: 

INFO:root:SPLADE index saved to /work/sandbox/splade/index/bert-base-spanish-wwm-cased_resumen_full


Successfully processed 47514 documents in chunks!
Completed processing resumen documents in 789.79 seconds. Estimated time remaining: 0 minutes, 0 seconds.

Total execution time: 1779.36 seconds
SPLADE indexing completed successfully!


In [ ]:
# Improved

In [2]:
import sys
from pathlib import Path
import pickle
import time
import logging
import numpy as np
import torch
from scipy.sparse import csr_matrix, save_npz
from transformers import AutoModelForMaskedLM, AutoTokenizer
from llama_index.core import Document
from typing import List, Union

# Add parent directory to Python path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents

class SpladeSparseVectorizer:
    """
    SPLADE (SParse Lexical AnD Expansion) vectorizer for Spanish documents.
    Uses a pre-trained Spanish BERT model to create sparse document representations.
    """
    
    def __init__(self, model_name="dccuchile/bert-base-spanish-wwm-cased", 
                 max_length=256, batch_size=2, device=None, 
                 sparsity_threshold=0.5, top_k=200):
        """
        Initialize SPLADE vectorizer with aggressive sparsity controls.
        
        Args:
            model_name: HuggingFace model name for Spanish BERT
            max_length: Maximum sequence length for tokenization
            batch_size: Batch size for processing documents
            device: Device to run the model on (auto-detected if None)
            sparsity_threshold: Minimum value threshold for sparse features (increased)
            top_k: Maximum number of features to keep per document (NEW)
        """
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.sparsity_threshold = sparsity_threshold
        self.top_k = top_k
        
        # Set device
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        
        print(f"Using device: {self.device}")
        print(f"Sparsity settings: threshold={sparsity_threshold}, top_k={top_k}")
        
        # Load tokenizer and model with memory optimizations
        print(f"Loading tokenizer and model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Load model with consistent precision
        if self.device.type == 'cuda':
            self.model = AutoModelForMaskedLM.from_pretrained(
                model_name, 
                torch_dtype=torch.float32
            )
        else:
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)
            
        self.model.to(self.device)
        self.model.eval()
        
        # Vocabulary size
        self.vocab_size = len(self.tokenizer.vocab)
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Model loaded with batch_size={batch_size}, max_length={max_length}")
        
        # Document statistics
        self.fitted = False
        
    def _apply_sparsity_controls(self, doc_weights):
        """
        Apply aggressive sparsity controls to document weights.
        
        Args:
            doc_weights: Tensor of document weights [vocab_size]
            
        Returns:
            Sparse indices and values
        """
        # Method 1: Top-K selection (most important)
        top_k_values, top_k_indices = torch.topk(doc_weights, 
                                                 k=min(self.top_k, len(doc_weights)), 
                                                 largest=True)
        
        # Method 2: Apply threshold to top-K values
        threshold_mask = top_k_values > self.sparsity_threshold
        
        if threshold_mask.any():
            final_indices = top_k_indices[threshold_mask]
            final_values = top_k_values[threshold_mask]
            return final_indices, final_values
        else:
            # If no values meet threshold, keep only top 10 values
            top_minimal = min(10, len(top_k_values))
            return top_k_indices[:top_minimal], top_k_values[:top_minimal]
    
    def _extract_splade_features(self, texts: List[str]) -> csr_matrix:
        """
        Extract SPLADE sparse features from texts with aggressive sparsity.
        
        Args:
            texts: List of text strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        # Use lists to store sparse matrix components efficiently
        rows, cols, data = [], [], []
        current_row = 0
        
        print(f"Processing {len(texts)} texts in batches of {self.batch_size}...")
        print(f"Using sparsity controls: threshold={self.sparsity_threshold}, top_k={self.top_k}")
        batch_start_time = time.time()
        
        # Track sparsity statistics
        total_features = 0
        total_possible = 0
        
        # Process in batches with progress tracking
        for batch_idx in range(0, len(texts), self.batch_size):
            batch_texts = texts[batch_idx:batch_idx + self.batch_size]
            
            # Enhanced progress tracking
            current_batch = batch_idx // self.batch_size + 1
            total_batches = (len(texts) - 1) // self.batch_size + 1
            
            if current_batch % max(1, total_batches // 20) == 0 or current_batch == 1:
                progress = (current_batch / total_batches) * 100
                print(f"Progress: {progress:.1f}% - Batch {current_batch}/{total_batches}")
                
                # Show current sparsity stats
                if total_possible > 0:
                    current_sparsity = 1 - (total_features / total_possible)
                    avg_features_per_doc = total_features / current_row if current_row > 0 else 0
                    print(f"Current sparsity: {current_sparsity:.4f}, avg features per doc: {avg_features_per_doc:.1f}")
                
                # Estimate remaining time
                if current_batch > 1:
                    elapsed = time.time() - batch_start_time
                    avg_time_per_batch = elapsed / (current_batch - 1)
                    remaining_batches = total_batches - current_batch
                    est_remaining = avg_time_per_batch * remaining_batches
                    minutes, seconds = divmod(est_remaining, 60)
                    print(f"Estimated time remaining: {int(minutes)}m {int(seconds)}s")
            
            try:
                # Tokenize batch
                inputs = self.tokenizer(
                    batch_texts,
                    max_length=self.max_length,
                    padding=True,
                    truncation=True,
                    return_tensors="pt"
                ).to(self.device)
                
                with torch.no_grad():
                    # Get model outputs
                    outputs = self.model(**inputs)
                    logits = outputs.logits  # [batch_size, seq_len, vocab_size]
                    
                    # Apply log(1 + ReLU) activation to get sparse weights
                    sparse_weights = torch.log1p(torch.relu(logits))
                    
                    # Max pooling over sequence dimension
                    doc_sparse = torch.max(sparse_weights, dim=1)[0]
                    
                    # Process each document in the batch
                    for batch_doc_idx, doc_weights in enumerate(doc_sparse):
                        # Apply aggressive sparsity controls
                        indices, values = self._apply_sparsity_controls(doc_weights)
                        
                        if len(indices) > 0:
                            # Convert to CPU and add to sparse matrix components
                            indices_cpu = indices.cpu().numpy().astype(np.int32)
                            values_cpu = values.cpu().numpy().astype(np.float32)
                            
                            # Add to sparse matrix components
                            rows.extend([current_row] * len(indices_cpu))
                            cols.extend(indices_cpu.tolist())
                            data.extend(values_cpu.tolist())
                            
                            # Update statistics
                            total_features += len(indices_cpu)
                        
                        total_possible += self.vocab_size
                        current_row += 1
                
                # Clear GPU memory after each batch
                del inputs, outputs, logits, sparse_weights, doc_sparse
                torch.cuda.empty_cache() if torch.cuda.is_available() else None
                
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"GPU memory error at batch {batch_idx // self.batch_size + 1}. Trying to continue...")
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
                    # Try processing with smaller batch size
                    return self._extract_splade_features_fallback(texts)
                else:
                    raise e
        
        # Create sparse matrix from components
        if rows:
            rows = np.array(rows, dtype=np.int32)
            cols = np.array(cols, dtype=np.int32)
            data = np.array(data, dtype=np.float32)
            
            sparse_matrix = csr_matrix((data, (rows, cols)), 
                                     shape=(len(texts), self.vocab_size),
                                     dtype=np.float32)
        else:
            sparse_matrix = csr_matrix((len(texts), self.vocab_size), dtype=np.float32)
        
        # Final statistics
        final_sparsity = 1 - (sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]))
        avg_features = sparse_matrix.nnz / sparse_matrix.shape[0] if sparse_matrix.shape[0] > 0 else 0
        
        print(f"Created sparse matrix with shape: {sparse_matrix.shape}")
        print(f"Non-zero elements: {sparse_matrix.nnz}")
        print(f"Final sparsity: {final_sparsity:.4f}")
        print(f"Average features per document: {avg_features:.1f}")
        
        return sparse_matrix
    
    def _extract_splade_features_fallback(self, texts: List[str]) -> csr_matrix:
        """
        Fallback method with smaller batch size for memory issues.
        """
        print("Using fallback method with batch size 1...")
        original_batch_size = self.batch_size
        self.batch_size = 1
        
        try:
            result = self._extract_splade_features(texts)
            return result
        finally:
            self.batch_size = original_batch_size
    
    def fit(self, documents: List[Union[Document, str]]):
        """
        Fit the SPLADE vectorizer to documents.
        
        Args:
            documents: List of Document objects or strings
        """
        print("Extracting text from documents...")
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        
        print(f"Fitting SPLADE vectorizer on {len(texts)} documents...")
        
        # For SPLADE, we don't need to compute global statistics
        self.fitted = True
        print("SPLADE vectorizer fitted successfully.")
        
        return self
    
    def transform(self, documents: List[Union[Document, str]]) -> csr_matrix:
        """
        Transform documents to SPLADE sparse representations.
        
        Args:
            documents: List of Document objects or strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        if not self.fitted:
            raise ValueError("Vectorizer must be fitted before transforming documents.")
        
        if isinstance(documents, str):
            documents = [documents]
        
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        
        print(f"Transforming {len(texts)} documents to SPLADE representations...")
        sparse_matrix = self._extract_splade_features(texts)
        
        print(f"Generated sparse matrix with shape: {sparse_matrix.shape}")
        print(f"Sparsity: {1 - sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]):.4f}")
        
        return sparse_matrix

def save_index(index_path: Path, vectorizer: SpladeSparseVectorizer, embeddings: csr_matrix, documents: List):
    """
    Save SPLADE index components to disk.
    """
    index_path.mkdir(parents=True, exist_ok=True)
    
    # Save vectorizer state (including sparsity settings)
    vectorizer_state = {
        'model_name': vectorizer.model_name,
        'max_length': vectorizer.max_length,
        'batch_size': vectorizer.batch_size,
        'vocab_size': vectorizer.vocab_size,
        'fitted': vectorizer.fitted,
        'sparsity_threshold': vectorizer.sparsity_threshold,
        'top_k': vectorizer.top_k
    }
    
    with open(index_path / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer_state, f)
    
    # Save embeddings
    save_npz(index_path / "embeddings.npz", embeddings)
    
    # Save documents
    with open(index_path / "documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    
    logging.info(f"SPLADE index saved to {index_path}")

def load_splade_vectorizer(index_path: Path) -> SpladeSparseVectorizer:
    """
    Load a saved SPLADE vectorizer.
    """
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer_state = pickle.load(f)
    
    # Recreate vectorizer with sparsity settings
    vectorizer = SpladeSparseVectorizer(
        model_name=vectorizer_state['model_name'],
        max_length=vectorizer_state['max_length'],
        batch_size=vectorizer_state['batch_size'],
        sparsity_threshold=vectorizer_state.get('sparsity_threshold', 0.5),
        top_k=vectorizer_state.get('top_k', 200)
    )
    vectorizer.fitted = vectorizer_state['fitted']
    
    return vectorizer

def main(file: str, model_name: str = "dccuchile/bert-base-spanish-wwm-cased", 
         max_length: int = 512, batch_size: int = 2, chunk_size: int = 5000,
         sparsity_threshold: float = 0.5, top_k: int = 200):
    """
    Main function to create SPLADE indices with aggressive sparsity controls.
    
    Args:
        file: Base filename for the document files
        model_name: HuggingFace model name for Spanish BERT
        max_length: Maximum sequence length
        batch_size: Batch size for processing
        chunk_size: Process documents in chunks
        sparsity_threshold: Minimum value threshold for features (increased default)
        top_k: Maximum features per document (NEW parameter)
    """
    start = time.time()
    print("Starting SPLADE indexing process with aggressive sparsity controls...")
    print(f"Settings: max_length={max_length}, batch_size={batch_size}, chunk_size={chunk_size}")
    print(f"Sparsity: threshold={sparsity_threshold}, top_k={top_k}")
    
    # Setup logging
    logging.basicConfig(level=logging.INFO)
    
    print("Setting up file paths...")
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')
    
    # Create index paths with sparsity info
    model_suffix = model_name.split('/')[-1]
    sparsity_suffix = f"_t{sparsity_threshold}_k{top_k}"
    index_path_texto = Path(f'/work/sandbox/splade/index/{model_suffix}_texto_sparse{sparsity_suffix}')
    index_path_resumen = Path(f'/work/sandbox/splade/index/{model_suffix}_resumen_sparse{sparsity_suffix}')
    
    print("Loading documents...")
    documents_texto = load_documents(documents_path_texto)
    documents_resumen = load_documents(documents_path_resumen)
    
    if documents_texto is None or documents_resumen is None:
        logging.error("Failed to load one or both document sets. Exiting.")
        return
    
    print(f"Processing documents - texto: {len(documents_texto)}, resumen: {len(documents_resumen)}")
    
    datasets = [
        ("texto", documents_texto, index_path_texto),
        ("resumen", documents_resumen, index_path_resumen)
    ]
    
    for i, (name, docs, index_path) in enumerate(datasets, 1):
        print(f"\nProcessing {name} documents ({i}/{len(datasets)})...")
        step_start = time.time()
        
        try:
            process_dataset_in_chunks(docs, index_path, model_name, max_length, 
                                    batch_size, chunk_size, name, 
                                    sparsity_threshold, top_k)
        except Exception as e:
            print(f"Error processing {name} documents: {str(e)}")
            continue
        
        step_end = time.time()
        print(f"Completed {name} in {step_end - step_start:.2f} seconds")
    
    end = time.time()
    print(f"\nTotal execution time: {end - start:.2f} seconds")
    print("SPLADE indexing completed!")

def process_dataset_in_chunks(documents, index_path, model_name, max_length, batch_size, 
                            chunk_size, dataset_name, sparsity_threshold, top_k):
    """
    Process dataset in chunks with sparsity controls.
    """
    from scipy.sparse import vstack as sparse_vstack
    
    total_docs = len(documents)
    num_chunks = (total_docs + chunk_size - 1) // chunk_size
    
    print(f"Processing {total_docs} documents in {num_chunks} chunks")
    
    all_embeddings = []
    chunk_start_time = time.time()
    
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_docs)
        chunk_docs = documents[start_idx:end_idx]
        
        print(f"\nChunk {chunk_idx + 1}/{num_chunks}: Processing documents {start_idx}-{end_idx}")
        
        # Initialize vectorizer with sparsity controls
        vectorizer = SpladeSparseVectorizer(
            model_name=model_name,
            max_length=max_length,
            batch_size=batch_size,
            sparsity_threshold=sparsity_threshold,
            top_k=top_k
        )
        
        # Process chunk
        vectorizer.fit(chunk_docs)
        chunk_embeddings = vectorizer.transform(chunk_docs)
        all_embeddings.append(chunk_embeddings)
        
        # Show chunk statistics
        chunk_sparsity = 1 - (chunk_embeddings.nnz / (chunk_embeddings.shape[0] * chunk_embeddings.shape[1]))
        print(f"Chunk sparsity: {chunk_sparsity:.4f}")
        
        # Clean up
        del vectorizer, chunk_embeddings
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        # Time estimation
        chunk_elapsed = time.time() - chunk_start_time
        avg_time_per_chunk = chunk_elapsed / (chunk_idx + 1)
        remaining_chunks = num_chunks - (chunk_idx + 1)
        est_remaining = avg_time_per_chunk * remaining_chunks
        minutes, seconds = divmod(est_remaining, 60)
        print(f"Estimated time remaining: {int(minutes)}m {int(seconds)}s")
    
    # Combine all embeddings
    print("Combining embeddings...")
    final_embeddings = sparse_vstack(all_embeddings)
    
    # Final statistics
    final_sparsity = 1 - (final_embeddings.nnz / (final_embeddings.shape[0] * final_embeddings.shape[1]))
    avg_features = final_embeddings.nnz / final_embeddings.shape[0]
    
    print(f"Final combined statistics:")
    print(f"Shape: {final_embeddings.shape}")
    print(f"Non-zero elements: {final_embeddings.nnz}")
    print(f"Sparsity: {final_sparsity:.4f}")
    print(f"Average features per document: {avg_features:.1f}")
    
    # Create final vectorizer for saving
    final_vectorizer = SpladeSparseVectorizer(
        model_name=model_name,
        max_length=max_length,
        batch_size=batch_size,
        sparsity_threshold=sparsity_threshold,
        top_k=top_k
    )
    final_vectorizer.fit(documents[:10])
    
    # Save index
    save_index(index_path, final_vectorizer, final_embeddings, documents)
    
    # Cleanup
    del all_embeddings, final_embeddings, final_vectorizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [3]:
main('OEB', "dccuchile/bert-base-spanish-wwm-cased", 512, 2)

Starting SPLADE indexing process with aggressive sparsity controls...
Settings: max_length=512, batch_size=2, chunk_size=5000
Sparsity: threshold=0.5, top_k=200
Setting up file paths...
Loading documents...


INFO:root:Successfully loaded 47514 documents from /work/data/processed/OEB_texto.pkl
INFO:root:Successfully loaded 47514 documents from /work/data/processed/OEB_resumen.pkl


Processing documents - texto: 47514, resumen: 47514

Processing texto documents (1/2)...
Processing 47514 documents in 10 chunks

Chunk 1/10: Processing documents 0-5000
Using device: cuda
Sparsity settings: threshold=0.5, top_k=200
Loading tokenizer and model: dccuchile/bert-base-spanish-wwm-cased
Vocabulary size: 31002
Model loaded with batch_size=2, max_length=512
Extracting text from documents...
Fitting SPLADE vectorizer on 5000 documents...
SPLADE vectorizer fitted successfully.
Transforming 5000 documents to SPLADE representations...
Processing 5000 texts in batches of 2...
Using sparsity controls: threshold=0.5, top_k=200
Progress: 0.0% - Batch 1/2500
Progress: 5.0% - Batch 125/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estimated time remaining: 0m 29s
Progress: 10.0% - Batch 250/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estimated time remaining: 0m 25s
Progress: 15.0% - Batch 375/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estima

INFO:root:SPLADE index saved to /work/sandbox/splade/index/bert-base-spanish-wwm-cased_texto_sparse_t0.5_k200


Completed texto in 292.30 seconds

Processing resumen documents (2/2)...
Processing 47514 documents in 10 chunks

Chunk 1/10: Processing documents 0-5000
Using device: cuda
Sparsity settings: threshold=0.5, top_k=200
Loading tokenizer and model: dccuchile/bert-base-spanish-wwm-cased
Vocabulary size: 31002
Model loaded with batch_size=2, max_length=512
Extracting text from documents...
Fitting SPLADE vectorizer on 5000 documents...
SPLADE vectorizer fitted successfully.
Transforming 5000 documents to SPLADE representations...
Processing 5000 texts in batches of 2...
Using sparsity controls: threshold=0.5, top_k=200
Progress: 0.0% - Batch 1/2500
Progress: 5.0% - Batch 125/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estimated time remaining: 0m 32s
Progress: 10.0% - Batch 250/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estimated time remaining: 0m 25s
Progress: 15.0% - Batch 375/2500
Current sparsity: 0.9935, avg features per doc: 200.0
Estimated time remaini

INFO:root:SPLADE index saved to /work/sandbox/splade/index/bert-base-spanish-wwm-cased_resumen_sparse_t0.5_k200


Completed resumen in 258.49 seconds

Total execution time: 554.59 seconds
SPLADE indexing completed!
